# Fine-tune Qwen2.5-7B-Instruct to chat like Kaushik

LoRA SFT on the `kaydee647-chat-dataset` Kaggle dataset (`train.jsonl` / `val.jsonl` from `build_dataset.py`). Loss is computed only on Kaushik's reply.

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **on**, secret `hf_token` (HF write token).

Training runs as a script under `torchrun` so both T4s are used; a notebook cell can only drive one GPU. Don't load the model in this notebook before training, or it holds GPU memory the training processes need.

For the full run use **Save Version → Save & Run All (Commit)**: it runs unattended for up to 12 h and keeps `/kaggle/working`. Checkpoints every 250 steps; re-running resumes from the last one.

Order: config → peek data → write `train_ddp.py` → 20-step smoke test → full run → evaluate → upload adapter to HF.

In [ ]:
!pip install -q unsloth trl

In [ ]:
import json
import os

from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
DATA_DIR = "/kaggle/input/datasets/kaushikdey2026/kaydee647-chat-dataset"
WORK = "/kaggle/working/outputs"
OUT = f"{WORK}/lora"
EPOCHS = 3
HF_REPO_NAME = "kaydee-qwen2.5-7b-lora"

os.environ.update(MODEL=MODEL, DATA_DIR=DATA_DIR, WORK=WORK, EPOCHS=str(EPOCHS))  # read by train_ddp.py
hf_token = UserSecretsClient().get_secret("hf_token")
os.environ["HF_TOKEN"] = hf_token  # authenticated downloads in the torchrun workers too
!nvidia-smi -L

## Data

In [ ]:
ds = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl", "validation": f"{DATA_DIR}/val.jsonl"})
print(ds)

for row in ds["train"].shuffle(seed=0).select(range(2)):
    print("-" * 60)
    for t in row["prompt"]:
        print(f"{t['role']:>9}: {t['content']!r}")
    print(f"   TARGET: {row['completion'][0]['content']!r}")

## Training script

One process per GPU, each with a full 4-bit copy of the model (DDP). Effective batch stays 16: `2 per GPU x 4 accumulation x 2 GPUs`. With one GPU it switches to 8 accumulation steps automatically.

In [ ]:
%%writefile train_ddp.py
import json
import os

from unsloth import FastLanguageModel  # must be imported before transformers/trl

import torch
from datasets import load_dataset
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

MODEL = os.environ["MODEL"]
DATA_DIR = os.environ["DATA_DIR"]
EPOCHS = float(os.environ.get("EPOCHS", 3))
MAX_STEPS = int(os.environ.get("MAX_STEPS", -1))
OUT_DIR = os.environ["WORK"] + ("/smoke" if MAX_STEPS > 0 else "")  # smoke run never gets resumed by the full run
MAX_SEQ_LEN = 2048
BATCH = 2          # per GPU; 7B 4-bit on a 16 GB T4
TOTAL_BATCH = 16
EVAL_SAMPLES = 400  # full val set every eval would add hours over ~5.8k steps
EVAL_EVERY = 250

rank = int(os.environ.get("LOCAL_RANK", 0))
world = int(os.environ.get("WORLD_SIZE", 1))
bf16 = torch.cuda.get_device_capability(rank)[0] >= 8  # T4 (sm75) only emulates bf16, which is slow

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, device_map={"": rank},
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

ds = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl", "validation": f"{DATA_DIR}/val.jsonl"})

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"].shuffle(seed=0).select(range(min(EVAL_SAMPLES, len(ds["validation"])))),
    args=SFTConfig(
        output_dir=OUT_DIR,
        max_length=MAX_SEQ_LEN,
        completion_only_loss=True,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=max(1, TOTAL_BATCH // (BATCH * world)),
        learning_rate=2e-4,
        num_train_epochs=EPOCHS,
        max_steps=MAX_STEPS,
        lr_scheduler_type="cosine",
        warmup_steps=0.03,
        weight_decay=0.01,
        fp16=not bf16,
        bf16=bf16,
        optim="adamw_8bit",
        eval_strategy="steps",
        eval_steps=EVAL_EVERY,
        logging_steps=10,
        save_steps=EVAL_EVERY,
        save_total_limit=2,
        ddp_find_unused_parameters=False,
        report_to="none",
        seed=0,
    ),
)

if trainer.is_world_process_zero():
    b = trainer.train_dataset[0]
    if "labels" in b:
        kept = [t for t, l in zip(b["input_ids"], b["labels"]) if l != -100]
        print(f"[sanity] {len(b['input_ids'])} tokens, {len(kept)} trained on: {tokenizer.decode(kept)[:300]!r}")
    else:
        print("[sanity] no labels column, columns are", list(b))

last = get_last_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
if last and trainer.is_world_process_zero():
    print("resuming from", last)
trainer.train(resume_from_checkpoint=last)
metrics = trainer.evaluate()  # every rank must call this under DDP

if trainer.is_world_process_zero():
    print(metrics)
    model.save_pretrained(f"{OUT_DIR}/lora")
    tokenizer.save_pretrained(f"{OUT_DIR}/lora")
    with open(f"{OUT_DIR}/lora/log_history.json", "w") as f:
        json.dump(trainer.state.log_history, f, indent=1)
    print("saved", f"{OUT_DIR}/lora")

## Smoke test (20 steps)

Check in the output:
- Unsloth's banner says `Num GPUs used = 2`, and total batch size is 16.
- `[sanity]` text is only short replies ending in `<|im_end|>`, no system prompt or friend messages.
- No out-of-memory error. If it OOMs, set `BATCH = 1` in the script.
- Seconds per step: full run time is about `steps_per_epoch x 3 x s/step`.

If torchrun fails with this Unsloth version, fall back to one GPU: `!python train_ddp.py` with `EPOCHS = 2` (about 9 h).

In [ ]:
!MAX_STEPS=20 torchrun --nproc_per_node 2 train_ddp.py

## Full run

Smoke test measured ~4.2 s/step on 2x T4: ~1,920 steps/epoch x 3 = ~5,760 steps, about 6.5-7 h plus ~25 short evals. Run it via Save & Run All (Commit). Checkpoint + eval every 250 steps (~18 min).

In [ ]:
!torchrun --nproc_per_node 2 train_ddp.py

## Evaluate

Loads the trained adapter in this notebook (training processes have exited, so the GPUs are free).

In [ ]:
# Imported here, not at the top: loading Unsloth before training would hold GPU memory the torchrun workers need.
from unsloth import FastLanguageModel
import pandas as pd

h = pd.DataFrame(json.load(open(f"{OUT}/log_history.json")))
print(h[["step", "loss"]].dropna().iloc[::5].to_string(index=False))
print(h[["step", "eval_loss"]].dropna().to_string(index=False))

model, tokenizer = FastLanguageModel.from_pretrained(OUT, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)

In [ ]:
def reply(messages, max_new_tokens=128, temperature=0.8):
    """messages: chat history ending on a user turn -> generated assistant reply."""
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, max_length=None, do_sample=True, temperature=temperature, top_p=0.9
    )
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


for row in ds["validation"].shuffle(seed=1).select(range(10)):
    print("-" * 60)
    for t in row["prompt"][1:]:
        print(f"{t['role']:>9}: {t['content']!r}")
    print(f"    MODEL: {reply(row['prompt'])!r}")
    print(f"   ACTUAL: {row['completion'][0]['content']!r}")

In [ ]:
SYSTEM = "You are Kaushik Dey, chatting with {friend} on Instagram. Reply the way Kaushik would."
for friend, text in [("Swapnil Agarwal", "Aur btao kya chal rha h"), ("Avantika", "Weekend pe kya plan hai?"), ("Swapnil Agarwal", "What do you do for work?")]:
    msgs = [{"role": "system", "content": SYSTEM.format(friend=friend)}, {"role": "user", "content": text}]
    print(f"[{friend}] {text}")
    for _ in range(3):
        print("   ->", reply(msgs).replace("\n", " / "))

## Upload adapter to Hugging Face (private)

Trained on private chats, so the repo is always private. GGUF conversion is done locally afterwards:

```bash
hf download <you>/kaydee-qwen2.5-7b-lora --local-dir outputs/lora-7b
uv run python export_gguf.py --adapter outputs/lora-7b --base Qwen/Qwen2.5-7B-Instruct --name kaydee-qwen2.5-7b
```

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
repo = f"{api.whoami()['name']}/{HF_REPO_NAME}"
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=repo)
print(f"https://huggingface.co/{repo}")